In [ ]:
# %% [markdown]
# # 🎬 VideoRasterization Pipeline (Notebook Edition)
# ---
# Central notebook for testing each pipeline step separately.
#
# **Flow**
# 1️⃣ Setup (imports, threads, paths)  
# 2️⃣ Select input video  
# 3️⃣ Extract frames via FFmpeg  
# 4️⃣ Choose & load AI model (Zhang)  
# 5️⃣ Run colorizer (CPU for now)  
# 6️⃣ Optional: rebuild video / report
# ---

# %%
# ## 1️⃣ Setup: imports, paths, environment

from importlib import import_module
from pathlib import Path
import sys, os, multiprocessing as mp
import imageio_ffmpeg

# --- max CPU threads ---
LOGICAL = mp.cpu_count() or 8
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(var, str(LOGICAL))

# --- set repo root ---
ROOT = Path.cwd()   # assume notebook in repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"[ok] repo root: {ROOT}")
print(f"[ok] threads: {LOGICAL}")

# %%
# ## 2️⃣ Input: choose video file

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
input_selector = import_module("tools.input_selector")

# interactively select
video_path = input_selector.get_input_video_path(allowed_exts=VIDEO_EXTS)

# or hard-code for speed:
# video_path = ROOT / "demoGreyscale.mp4"

print(f"[ok] selected video: {video_path}")

# %%
# ## 3️⃣ Extract frames using FFmpeg

temp_root = ROOT / "temp"
temp_root.mkdir(parents=True, exist_ok=True)

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_tools = import_module("tools.FFmpeg.FFmpeg_utilization")

frames_dir = ffmpeg_tools.extract_frames(ffmpeg_path, video_path, temp_root)

if not frames_dir or not Path(frames_dir).exists():
    raise RuntimeError("Frame extraction failed.")
print(f"[ok] extracted frames dir: {frames_dir}")

# %%
# ## 4️⃣ Choose and load AI model (Zhang for now)

model_selector = import_module("tools.model_selector")
models_root = ROOT / "tools" / "AImodels"

model_name = model_selector.select_model(models_root)
if not model_name:
    raise RuntimeError("No AI models found.")

print(f"[ok] selected model: {model_name}")

zhang_variant = "siggraph17"   # or "eccv16"
use_gpu = False                # CPU for now

# %%
# ## 5️⃣ Colorize frames

frames_path = Path(frames_dir)
color_dir = frames_path.parent / f"{frames_path.name}_colorized"
color_dir.mkdir(parents=True, exist_ok=True)

model_selector.run_colorizer(
    model_name=model_name,
    frames_dir=frames_path,
    color_dir=color_dir,
    models_dir=ROOT / "models",
    zhang_variant=zhang_variant,  # selector maps to 'variant'
    preview=False,
    use_gpu=use_gpu,
    batch_size=None,              # auto
    num_threads=None,             # auto = logical cores
    input_size=256,               # 224 slightly faster
    progress=True,
    prefetch_workers=None,        # auto
    save_workers=4,               # 0-4 depending on disk speed
)

print(f"[ok] colorization complete: {color_dir}")

# %%
# ## 6️⃣ Optional: rebuild video / generate report (future)
# Add temporal smoothing, FFmpeg merge, or report generation here.

# Example placeholders:
# ffmpeg_tools.rebuild_video(...)
# report = import_module("tools.preview_report")
# report.generate_report(...)

print("[done] pipeline finished. frames ready.")
